In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-ml kailash-pact kailash-nexus kailash kailash-kaizen

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT10 — Exercise 6: Regulatory Compliance Automation with PACT
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build automated compliance checking -- define regulation
#   schemas as Signatures, implement a ComplianceAuditAgent, trigger
#   audits from DriftMonitor alerts, and deploy via Nexus.
#
# TASKS:
#   1. Define compliance schema using Signature for EU AI Act, MAS TRM,
#      SG AI Verify, PDPA
#   2. Implement ComplianceAuditAgent that inspects governance artefacts
#   3. Run audit on governed pipeline, generate remediation recommendations
#   4. Continuous compliance: DriftMonitor alert triggers audit agent
#   5. Deploy compliance checker as Nexus endpoint
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import json
import os
import tempfile
import uuid
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl

from kaizen import Signature, InputField, OutputField
from kaizen.core import Agent as BaseAgent
from kailash.db.connection import ConnectionManager
from kailash_ml.engines.drift_monitor import DriftMonitor
from kailash.trust import ConfidentialityLevel, TrustPosture
from pact import GovernanceEngine, PactGovernedAgent, compile_org, load_org_yaml
from pact.governance import RoleClearance, RoleEnvelope
from kailash.trust.pact.config import (
    ConstraintEnvelopeConfig,
    OperationalConstraintConfig,
    TemporalConstraintConfig,
    DataAccessConstraintConfig,
    CommunicationConstraintConfig,
)
from nexus import Nexus

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()



## TASK 1: Define compliance schemas using Signature


In [ ]:
print("=== Compliance Schemas (Regulation -> Assertion Predicates) ===\n")


class EUAIActComplianceSignature(Signature):
    """EU AI Act compliance check for high-risk AI systems."""

    # TODO: InputFields for model_card_exists, bias_audit_conducted,
    # TODO: human_oversight_defined, risk_category (str), audit_trail_active,
    # TODO: drift_monitoring_active.
    # TODO: OutputFields for overall_status, article_verdicts, remediation_actions.
    ____


class MASTRMComplianceSignature(Signature):
    """MAS Technology Risk Management compliance check."""

    # TODO: InputFields for governance_framework_active, audit_trail_integrity,
    # TODO: model_risk_documented, change_management_logged, incident_response_defined.
    # TODO: OutputFields overall_status, section_verdicts, remediation_actions.
    ____


class SGAIVerifySignature(Signature):
    """Singapore AI Verify framework compliance check."""

    # TODO: InputFields for accountability_chain, fairness_audit, transparency_docs,
    # TODO: explainability_available, data_governance.
    # TODO: OutputFields overall_status, pillar_verdicts, remediation_actions.
    ____


class PDPAComplianceSignature(Signature):
    """Singapore PDPA (Personal Data Protection Act) compliance check."""

    # TODO: InputFields for consent_documented, data_minimisation, access_controls,
    # TODO: retention_policy, breach_notification.
    # TODO: OutputFields overall_status, obligation_verdicts, remediation_actions.
    ____


# TODO: Build a REGULATION_SCHEMAS dict mapping each regulation to a list
# TODO: of (assertion_name, predicate_string) pairs. Predicates use plain
# TODO: AND logic over artefact keys (e.g. "model_card_exists AND drift_monitoring_active").
REGULATION_SCHEMAS = ____

for reg_name, schema in REGULATION_SCHEMAS.items():
    print(f"{reg_name}:")
    for assertion_name, predicate in schema["assertions"]:
        print(f"  {assertion_name}: {predicate}")
    print()



## TASK 2: Implement ComplianceAuditAgent


In [ ]:
print(f"{'=' * 70}")
print(f"=== ComplianceAuditAgent ===")
print(f"{'=' * 70}\n")


def run_compliance_audit(artefacts: dict) -> dict:
    """Run compliance audit against all regulation schemas."""
    # TODO: For each regulation, evaluate every assertion (AND of artefact terms).
    # TODO: Collect per-assertion PASS/FAIL, build per-regulation status, gather
    # TODO: missing artefacts as remediation strings, set overall_status if any fail.
    ____


# TODO: Build a pipeline_artefacts dict containing each regulation's required
# TODO: keys. Set incident_response_defined / explainability_available /
# TODO: retention_policy to False to surface gaps in the audit.
pipeline_artefacts = ____



## TASK 3: Run audit on governed pipeline, generate remediation


In [ ]:
# TODO: Run run_compliance_audit on pipeline_artefacts.
audit_report = ____

print(f"Audit ID: {audit_report['audit_id']}")
print(f"Timestamp: {audit_report['timestamp']}")
print(f"Overall Status: {audit_report['overall_status']}\n")

# TODO: For each regulation print status, per-assertion PASS/FAIL, and
# TODO: any remediation strings.
____

print(f"=== Remediation Plan ===")
# TODO: Aggregate every regulation's remediation entries into a single list
# TODO: and print as a numbered plan; print "No remediation actions" if empty.
all_remediation = []
____



## TASK 4: Continuous compliance — DriftMonitor triggers audit


In [ ]:
print(f"\n{'=' * 70}")
print(f"=== Continuous Compliance Monitoring ===")
print(f"{'=' * 70}\n")

rng = np.random.default_rng(42)
reference_data = rng.normal(0, 1, (200, 10))
feature_names = [f"feature_{i}" for i in range(10)]


async def _setup_drift_monitor():
    # TODO: ConnectionManager(":memory:"), initialise, build DriftMonitor
    # TODO: at psi_threshold=0.2, set_reference with the polars wrapper.
    ____


drift_conn, monitor  = await _setup_drift_monitor()
def on_drift_detected(drift_report, pipeline_artefacts: dict) -> dict:
    """Callback: when drift is flagged, invalidate stale artefacts and audit."""
    # TODO: Copy artefacts; if drift detected, set bias_audit_conducted=False.
    # TODO: Re-run run_compliance_audit and return the new report.
    ____


# TODO: Simulate a stable week (no drift) and a drifted week, then run
# TODO: on_drift_detected on the drifted result and print the new audit verdict.
stable_data = ____
stable_report = ____
print(f"Week 1 (stable): drift={stable_report.overall_drift_detected}")
if not stable_report.overall_drift_detected:
    print(f"  No compliance audit triggered.\n")

drifted_data = ____
drift_report = ____
print(f"Week 2 (drift): drift={drift_report.overall_drift_detected}")

if drift_report.overall_drift_detected:
    print(f"  Drift detected -> triggering compliance audit...\n")
    drift_audit = ____
    print(f"  Audit Status: {drift_audit['overall_status']}")
    for reg_name, reg_result in drift_audit["regulations"].items():
        if reg_result["status"] != "COMPLIANT":
            print(f"  {reg_name}: {reg_result['status']}")
            for action in reg_result["remediation"]:
                print(f"    - {action}")

print(f"\nContinuous compliance loop:")
print(f"  DriftMonitor (PSI > 0.2) -> ComplianceAuditAgent -> Remediation Plan")
print(f"  This runs automatically; humans review remediation at approval gates.")



## TASK 5: Deploy compliance checker as Nexus endpoint


In [ ]:
print(f"\n{'=' * 70}")
print(f"=== Nexus Compliance Endpoint ===")
print(f"{'=' * 70}\n")

# TODO: Instantiate a Nexus app (will register the handler below).
app = ____


async def compliance_check_handler(model_id: str) -> dict:
    """Nexus endpoint: submit model_id, receive compliance report."""
    # TODO: Simulate metadata lookup with the existing pipeline_artefacts dict.
    # TODO: Run run_compliance_audit, attach model_id, return.
    ____


print(f"Nexus endpoint: POST /compliance/check")
print(f"  Input: {{'model_id': 'credit_fraud_detector_v1'}}")

# TODO: Call compliance_check_handler('credit_fraud_detector_v1') via asyncio.run.
demo_report = ____

print(f"  Output:")
print(f"    audit_id: {demo_report['audit_id']}")
print(f"    model_id: {demo_report['model_id']}")
print(f"    overall_status: {demo_report['overall_status']}")
print(f"    regulations checked: {list(demo_report['regulations'].keys())}")

# TODO: Define a YAML org for the compliance service with audit_admin and
# TODO: audit_agent (agent: true). Grant clearances at secret/confidential.
compliance_org_yaml = ____

# TODO: Write to temp, load, compile, build GovernanceEngine.
comp_file = ____
____
comp_loaded = ____
comp_engine = ____

comp_compiled = ____
comp_roles = ____
____

level_map = ____
# TODO: Grant clearances on the engine for both roles.
____

# TODO: Set an envelope for audit_agent with allowed actions
# TODO: [run_audit, read_model_card, read_drift_status, generate_report].
audit_env = ____
____

# TODO: Wrap audit_agent in PactGovernedAgent.
governed_audit = ____

print(f"\nCompliance agent governance:")
print(f"  Clearance: {governed_audit.context.effective_clearance_level}")
print(f"  Actions: {sorted(governed_audit.context.allowed_actions)}")

# TODO: Verify each action via comp_engine.verify_action and print ALLOW/BLOCK.
____

# TODO: Verify audit integrity and clean up the temp YAML file.
integrity = ____
print(f"\nAudit integrity: {'VALID' if integrity else 'BROKEN'}")

____

print(
    "\n--- Exercise 6 complete: regulatory compliance automation with PACT + Nexus ---"
)

